In [154]:
import pickle
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

import matplotlib.pyplot as plt
import seaborn as sns


In [155]:
with open("../data/01-result/drugs_df.pkl","rb") as f:
    drugs_df =pickle.load(f)
with open("../data/01-result/indications_df.pkl","rb") as f:
    indications_df = pickle.load(f)
with open("../data/01-result/trials_df.pkl","rb") as f:
    trials_df =pickle.load(f)
with open("../data/01-result/extended_diseases_df.pkl","rb") as f:
    diseases_df = pickle.load(f)

# Drop duplicates

In [156]:
print(f"Drugs duplicates (drug_id): {drugs_df[['drug_id']].duplicated().sum()}")
print(f"Diseases duplicates (disease_id): {diseases_df[['disease_id']].duplicated().sum()}")

drugs_df = drugs_df.drop_duplicates(subset="drug_id")
diseases_df = diseases_df.drop_duplicates(subset = "disease_id")

Drugs duplicates (drug_id): 0
Diseases duplicates (disease_id): 18


# Keep only drugs passed phase III or IV

They are proven safe

In [157]:
drugs_df = drugs_df[ (drugs_df["max_phase"] == "4.0") | (drugs_df["max_phase"] == "3.0")]

# Curate trials and indications

In [158]:
# Extract interesting data from clinical trials - dates and results
final_trials_df = trials_df.drop(columns=["annotationSection", "documentSection"]) # type: ignore
final_trials_df["nct_id"] = None 
final_trials_df["status"] = None
final_trials_df["phase"] = None
final_trials_df["success"] = None
final_trials_df["median_p_value"] = None
final_trials_df["p_value_list"] = None
for i, study in final_trials_df.iterrows():
    # TODO: choose better dates
    final_trials_df.loc[i, "nct_id"] = study["protocolSection"]["identificationModule"]["nctId"]

    status_module = study["protocolSection"]["statusModule"]
    final_trials_df.loc[i, "status"] = status_module["overallStatus"]

    # NOTE: for first date, take the earliest available date
    first_date = "9999" # largest by default 
    # NOTE: for last date - the last time it was heard of it
    last_date = "0000" # smallest by default
    for key, val in status_module.items():
        if "date" in key.lower():
            # if dictionary with the "date" field
            if "date" in val:
                first_date = min(first_date, val["date"])
                last_date = max(last_date, val["date"] )
            # if itself a date
            elif isinstance(val, str):
                first_date = min(first_date, val)
                last_date = max(last_date, val )
    final_trials_df.loc[i, "first_date"] = first_date
    final_trials_df.loc[i, "last_date"] = last_date
    
    # final_trials_df.loc[i, "first_date"] = status_module.get("startDateStruct",{"date":None})["date"]


    final_trials_df.loc[i, "end_date"] = status_module.get("completionDateStruct",{"date":None})["date"]
    final_trials_df.loc[i, "why_stopped"] = status_module.get("whyStopped", None)
    if "designModule" in study["protocolSection"]:
        phases = study["protocolSection"]["designModule"].get("phases",[])
        phases = [int(phase[-1]) for phase in phases if phase != "NA"]
        if phases:
            final_trials_df.loc[i, "phase"] = np.max(phases)
    # Get p-values
    # if "conditionBrowseModule" in study["derivedSection"]:
    #     found_mesh = [mesh['id'] for mesh in  study["derivedSection"]["conditionBrowseModule"]["meshes"]]
    #     if True in [mesh in mesh_ids for mesh in found_mesh]:
    if study["hasResults"]:
        measures = study["resultsSection"]["outcomeMeasuresModule"]["outcomeMeasures"]
        p_values = [measure["analyses"][0]["pValue"] for measure in measures if "analyses" in measure if "pValue" in measure["analyses"][0]]
        p_values = [float(''.join([ch for ch in p if ch.isdigit() or ch=="."])) for p in p_values]
        if len(p_values) > 0:
            final_trials_df.at[i, "p_value_list"] = p_values
final_trials_df = final_trials_df.drop(columns=["derivedSection","protocolSection"])
first_columns = ['nct_id','success',"median_p_value",'phase','status', 'hasResults','why_stopped']
final_trials_df = final_trials_df[first_columns + [c for c in final_trials_df.columns if c not in first_columns]]

# TODO: criteria for fail and success!
for i, study in final_trials_df.iterrows():
    if study["p_value_list"]:
        medp = np.median(study["p_value_list"])
        prop05 = (np.array(study["p_value_list"]) < 0.05).sum() / len(study["p_value_list"])
        conflicting = (np.min(study["p_value_list"]) < 0.05) and ((np.array(study["p_value_list"]) > 0.2).sum() / len(study["p_value_list"]) > 0.5)
        if medp <= 0.05 or (medp <= 0.1 and prop05 >= 0.5):
            final_trials_df.loc[i, "success"] = "success"
        # else: 
        #     final_trials_df.loc[i, "success"] = "fail"
        elif 0.05 < medp <= 0.20 or (0.10 < medp <= 0.50 and 0.1 <= prop05 < 0.5) or conflicting:
            final_trials_df.loc[i, "success"] = "unknown"
        elif medp > 0.2 and prop05 < 0.1:
            final_trials_df.loc[i, "success"] = "fail"

In [159]:
print(f"Trial duplicates (nct_id): {final_trials_df['nct_id'].duplicated().sum()}")
final_trials_df = final_trials_df.drop_duplicates(subset='nct_id')

Trial duplicates (nct_id): 19


In [160]:
# Add this trial info to corresponding indications
final_indications_df = indications_df.drop(columns=["drugind_id", "mesh_heading", "parent_molecule_chembl_id"])
final_indications_df["nct_evidence"] = None
final_indications_df["first_trial_date"] = "9999"
final_indications_df["last_trial_date"] = "0000"

for i, indication in final_indications_df.iterrows():
    # TODO: finish logic with success
    # add dates 
    if indication["nct_ids"]:
        success_list = []
        first_trial_date = "9999"
        last_trial_date = "0000"
        for nct_id in indication["nct_ids"]:
            success = final_trials_df.loc[final_trials_df["nct_id"]==nct_id,"success"].values
            if len(success)>0:
                if success[0] is not None:
                    success_list.append(success[0] + str(final_trials_df.loc[final_trials_df["nct_id"]==nct_id,"phase"].values[0]))
            first_date = final_trials_df.loc[final_trials_df["nct_id"]==nct_id, "first_date"]
            if not first_date.empty:
                first_date = first_date.values[0]
                if isinstance(first_date, str) and first_date:
                    first_trial_date = min(first_trial_date, first_date)
            
            last_date = final_trials_df.loc[final_trials_df["nct_id"]==nct_id, "last_date"]
            if not last_date.empty:
                last_date = last_date.values[0]
                if isinstance(last_date, str) and last_date:
                    last_trial_date = max(last_trial_date, last_date)
        if first_trial_date:
            final_indications_df.loc[i, "first_trial_date"] =  first_trial_date   
        if last_trial_date:
            final_indications_df.loc[i, "last_trial_date"] =  last_trial_date 
        final_indications_df.at[i, "nct_evidence"] = success_list
final_indications_df.rename(columns={"molecule_chembl_id":"drug_id"}, inplace=True)
first_columns = ['drug_id','efo_term','efo_id','mesh_id', 'max_phase_for_ind','nct_evidence']
final_indications_df= final_indications_df[first_columns + [c for c in final_indications_df.columns if c not in first_columns]]
final_indications_df["disease_id"] = final_indications_df["efo_id"]

In [161]:
print(f"Indication duplicates (disease_id, drug_id): {final_indications_df[['disease_id','drug_id']].duplicated().sum()}")
final_indications_df = final_indications_df.drop_duplicates(subset=['disease_id','drug_id'])

Indication duplicates (disease_id, drug_id): 24


# Estimate the first trial date

In [162]:
drugs_df["first_approval"] = drugs_df["first_approval"].apply(lambda x: str(int(x)) if pd.notna(x) else "9999")
dates_df = final_indications_df.groupby(["drug_id", "disease_id"], as_index=False)["first_trial_date"].min()
dates_df = dates_df.merge(drugs_df[["drug_id","first_approval"]], how="left", on = "drug_id").fillna("9999")
for i, row in dates_df.iterrows():
    # NOTE: ATTENTION
    # If no information about first trial date for this indication wasn't available 
    # - borrow the first approval date of the drug
    if dates_df.loc[i, "first_trial_date"] == "9999" and dates_df.loc[i, "first_approval"] != "9999":
        dates_df.loc[i, "first_trial_date"] = dates_df.loc[i, "first_approval"] 
dates_df["first_trial_date"].value_counts()
dates_df = dates_df.drop(columns=["first_approval"])
# replace absent dates (oldest by default) with "0000" - 
# - will take only newest for test
dates_df["first_trial_date"] = dates_df["first_trial_date"].apply( lambda x : "0000"  if x == "9999" else x )

# Set up the label

In [163]:
final_indications_df["overall_success"] = None

# Define what a success of a drug-disease combination is
for i, row in final_indications_df.iterrows():

    # If phase 4 is passed, it's definitely a success
    if final_indications_df.loc[i,"max_phase_for_ind"] == "4.0":
        print(f"Decided SUCCESS: passed phase 4")
        final_indications_df.at[i, "overall_success"] = True
    
    # If there's evidence about trials for this combination
    elif final_indications_df.loc[i,"nct_evidence"]:
        all_results = final_indications_df.loc[i,"nct_evidence"]
        highest_phase = np.max([int(result[-1]) for result in all_results if "None" not in result])
        latest_results = np.array([result[:-1] for result in all_results if "None" not in result and int(result[-1]) == highest_phase])
        nb_success = np.sum(latest_results == "success")
        nb_fail = np.sum(latest_results == "fail")
        if highest_phase >= 3 and nb_success > 2*nb_fail:
            print(f"Decided SUCCESS: highest phase {highest_phase} with {latest_results}")
            final_indications_df.at[i, "overall_success"] = True
        elif nb_fail > 2*nb_success:
            print(f"Decided FAIL: highest phase {highest_phase} with {latest_results}")
            final_indications_df.at[i, "overall_success"] = False
    
    # If still not decided
    if final_indications_df.at[i, "overall_success"] is None:
    # If only early stages were tried, and there were no news for a while
    # probably it was a fail / abandoned
        if final_indications_df.loc[i,"last_trial_date"] < "2020" \
            or  final_indications_df.loc[i,"last_trial_date"] == "0000":
            print(f"Decided FAIL: last heard { final_indications_df.loc[i,'last_trial_date']} with max phase {final_indications_df.loc[i,'max_phase_for_ind']}" )
            final_indications_df.at[i, "overall_success"] = False
        else:
            print(f"Decided NOTHING: last heard { final_indications_df.loc[i,'last_trial_date']} with max phase {final_indications_df.loc[i,'max_phase_for_ind']}" )

final_indications_df["overall_success"] = final_indications_df["overall_success"].astype("boolean")

first_columns = ["drug_id", "disease_id", "overall_success", "nct_evidence", "max_phase_for_ind"] 
final_indications_df= final_indications_df[first_columns + [c for c in final_indications_df.columns if c not in first_columns]]

# keep only indications for safe drugs
final_indications_df= final_indications_df[final_indications_df["drug_id"].isin(drugs_df["drug_id"])]

Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided FAIL: highest phase 2 with ['fail']
Decided FAIL: last heard 2017-03-09 with max phase 3.0
Decided NOTHING: last heard 2022-04-14 with max phase 3.0
Decided FAIL: last heard 2017-03-21 with max phase 2.0
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided NOTHING: last heard 2025-06-04 with max phase 1.0
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided SUCCESS: passed phase 4
Decided NOTHING: last heard 2024-12-03 with max phase 3.0
Decided NOTHING: last heard 2025-09-26 with max phase 3.0
Decided NOTHING: last heard 2023-09-28 with max phase 3.0
Decided SUCCESS: passed phase 4
Decided NOTHING: last heard 2020-02-20 with max phase 3.0
Decided NOTHING: last heard 2021-04-08 with max phase 2.0
Decided NOTHING: last he

In [164]:
print(f"Indication duplicates (disease_id, drug_id): {final_indications_df[['disease_id','drug_id']].duplicated().sum()}")
final_indications_df = final_indications_df.drop_duplicates(subset=['disease_id','drug_id'])

Indication duplicates (disease_id, drug_id): 0


In [165]:
print(f"{final_indications_df['overall_success'].notna().sum()} present out of {final_indications_df.shape[0]}")
final_indications_df["overall_success"].value_counts()

2905 present out of 5816


overall_success
False    1935
True      970
Name: count, dtype: Int64

# Feature engineering based on pathways

In [166]:
# # TODO: generate other features
# for i, row in merged_df.iterrows():
#     merged_df.at[i, "n_shared_pathways"] = len(set(merged_df.loc[i,"drug_pathways"]) and set(merged_df.loc[i,"disease_pathways"]))

# convert lists of IDs to space separated strings for TF-IDF
drugs_df["drug_path_str"] = drugs_df["drug_pathways"].apply(lambda x: " ".join(x))
diseases_df["disease_path_str"] = diseases_df["disease_pathways"].apply(lambda x: " ".join(x))
corpus = pd.concat([drugs_df['drug_path_str'], diseases_df['disease_path_str']], ignore_index=True)

# TF-IDF helps downweighting housekeeping pathways and highlights informative ones
vectorizer = TfidfVectorizer(token_pattern=r"[A-Za-z0-9_-]+")
tfidf_matrix = vectorizer.fit_transform(corpus)

# TODO: decide on nb of components
svd = TruncatedSVD(n_components=50, random_state=42)
latent = svd.fit_transform(tfidf_matrix)

n_drugs = len(drugs_df)
drug_features = latent[:n_drugs]
disease_features = latent[n_drugs:]

for i in range(drug_features.shape[1]):
    drugs_df[f"drug_path_{i}"] = drug_features[:,i]

for i in range(disease_features.shape[1]):
    diseases_df[f"disease_path_{i}"] = disease_features[:,i]

# Save results

In [167]:
with open("../data/02-result/drugs_df.pkl","wb") as f:
    pickle.dump(drugs_df,f)
with open("../data/02-result/diseases_df.pkl","wb") as f:
    pickle.dump(diseases_df,f)
with open("../data/02-result/trials_df.pkl","wb") as f:
    pickle.dump(final_trials_df,f)
with open("../data/02-result/indications_df.pkl","wb") as f:
    pickle.dump(final_indications_df,f)
with open("../data/02-result/dates_df.pkl","wb") as f:
    pickle.dump(dates_df,f)
